# Notebook 10 – Feature Selection

## 1. Relevant, irrelevant, and redundant features
Feature selection chooses a subset of inputs. Irrelevant features add noise; redundant features repeat information; useful features can improve efficiency and sometimes generalization. Selection must be fitted using training data only.

In [1]:
import pandas as pd
import numpy as np
from sklearn.datasets import make_classification

X_array, y_array = make_classification(
    n_samples=250, n_features=8, n_informative=4, n_redundant=2,
    random_state=42
)
X = pd.DataFrame(X_array, columns=[f"feature_{i}" for i in range(8)])
X["constant_feature"] = 1
y = pd.Series(y_array, name="target")
display(X.head())

,feature_0,feature_1,feature_2,feature_3,feature_4,feature_5,feature_6,feature_7,constant_feature
0,0.039164,-1.238672,0.140982,1.310073,-1.260908,-1.467260,-1.013488,1.501273,1
1,-0.999313,-0.463646,0.371553,-0.274228,1.281766,0.404555,2.618401,-0.664187,1
2,-1.512596,-0.018345,0.171399,-1.296502,-1.075626,-0.370284,-0.401921,0.186611,1
3,0.709462,-1.030499,-1.262842,0.123230,4.267169,-1.090368,-0.158879,2.582701,1
4,1.530732,0.494718,-0.382171,-1.186697,0.784212,0.739131,-0.745032,-0.438274,1


## 2. Correlation-based selection
Correlation can flag strongly related numeric features. It does not prove causation and may miss nonlinear relationships. Correlation-based filtering should be done using training data and domain knowledge.

In [2]:
corr = X.drop(columns="constant_feature").corr().abs()
display(corr.round(2))
upper = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))
high_corr = [col for col in upper.columns if (upper[col] > 0.90).any()]
print("Features with a correlation above 0.90 with an earlier column:", high_corr)

,feature_0,feature_1,feature_2,feature_3,feature_4,feature_5,feature_6,feature_7
feature_0,1.00,0.10,0.08,0.04,0.04,0.05,0.02,0.06
feature_1,0.10,1.00,0.39,0.01,0.18,0.09,0.43,0.08
feature_2,0.08,0.39,1.00,0.01,0.68,0.28,0.30,0.56
feature_3,0.04,0.01,0.01,1.00,0.00,0.02,0.02,0.02
feature_4,0.04,0.18,0.68,0.00,1.00,0.23,0.19,0.05
feature_5,0.05,0.09,0.28,0.02,0.23,1.00,0.15,0.94
feature_6,0.02,0.43,0.30,0.02,0.19,0.15,1.00,0.29
feature_7,0.06,0.08,0.56,0.02,0.05,0.94,0.29,1.00


Features with a correlation above 0.90 with an earlier column: ['feature_7']


## 3. Variance threshold
Features with no variance contain no information in the training data. Low variance alone does not necessarily mean a feature is useless, especially for rare but important categories.

In [3]:
from sklearn.feature_selection import VarianceThreshold
selector = VarianceThreshold(threshold=0.0)
X_var = selector.fit_transform(X)
print("Original feature count:", X.shape[1])
print("After removing constant features:", X_var.shape[1])
print("Removed:", X.columns[~selector.get_support()].tolist())

Original feature count: 9
After removing constant features: 8
Removed: ['constant_feature']


## 4. Univariate feature selection and mutual information
Univariate selection scores each feature separately against the target. Mutual information can capture nonlinear dependence but estimates can be noisy. Do not run selection using the test target.

In [4]:
from sklearn.feature_selection import SelectKBest, mutual_info_classif

selector = SelectKBest(score_func=mutual_info_classif, k=4)
X_selected = selector.fit_transform(X, y)
print("Selected features:", X.columns[selector.get_support()].tolist())

Selected features: ['feature_1', 'feature_4', 'feature_5', 'feature_6']


## 5. Recursive Feature Elimination (RFE) and feature importance
RFE repeatedly fits an estimator and removes weaker features. Model feature importance can help interpretation but may be biased by data characteristics. Use cross-validation and training-only selection when comparing models.

In [5]:
from sklearn.feature_selection import RFE
from sklearn.ensemble import RandomForestClassifier

model = RandomForestClassifier(n_estimators=100, random_state=42)
rfe = RFE(estimator=model, n_features_to_select=4)
rfe.fit(X, y)
print("RFE selected:", X.columns[rfe.support_].tolist())

model.fit(X, y)
importance = pd.Series(model.feature_importances_, index=X.columns).sort_values(ascending=False)
display(importance)

RFE selected: ['feature_4', 'feature_5', 'feature_6', 'feature_7']


feature_6           0.274837
feature_5           0.165350
feature_4           0.153722
feature_7           0.123036
feature_1           0.113454
feature_2           0.073149
feature_3           0.049012
feature_0           0.047439
constant_feature    0.000000
dtype: float64

**Output explanation:** These are demonstrations on synthetic data, and the scores are not evidence about the Telco dataset. In a real workflow, split first and perform feature selection within the training pipeline or cross-validation.